In [ ]:
!pip -q install pypdf sentence-transformers openai scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.4 MB/s eta 0:00:00


**Import** libraries

In [ ]:
import os
import re
import getpass
from pathlib import Path

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import files

Upload research-paper **PDFs**

In [ ]:
uploaded = files.upload()

pdf_paths = [Path(name) for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_paths:
    raise ValueError("Please upload at least one PDF file.")

print("Uploaded papers:")
for path in pdf_paths:
    print("-", path.name)

Saving IEEE-Research-Paper.pdf to IEEE-Research-Paper.pdf
Uploaded papers:
- IEEE-Research-Paper.pdf


Extract text with page **metadata**

In [ ]:
def extract_pdf_pages(pdf_path):
    """Return one record per PDF page."""
    reader = PdfReader(str(pdf_path))
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            pages.append({
                "paper": pdf_path.name,
                "page": page_number,
                "text": text
            })

    return pages

pages = []

for pdf_path in pdf_paths:
    extracted_pages = extract_pdf_pages(pdf_path)
    pages.extend(extracted_pages)
    print(f"{pdf_path.name}: extracted {len(extracted_pages)} text pages")

if not pages:
    raise ValueError(
        "No readable text was extracted. The PDF may be scanned and require OCR."
    )

IEEE-Research-Paper.pdf: extracted 6 text pages


In [ ]:
CHUNK_SIZE = 220       # Approximate words per chunk
CHUNK_OVERLAP = 40     # Shared words between consecutive chunks

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

chunks = []

for page in pages:
    page_chunks = chunk_text(page["text"])

    for index, chunk in enumerate(page_chunks, start=1):
        chunks.append({
            "id": len(chunks) + 1,
            "paper": page["paper"],
            "page": page["page"],
            "chunk_number": index,
            "text": chunk
        })

print(f"Created {len(chunks)} chunks from {len(pages)} pages.")
print("\nExample chunk:\n")
print(chunks[0]["text"][:700])

Created 26 chunks from 6 pages.

Example chunk:

Leveraging N8N Agent Platform and AI for Automated Job Candidate Screening 1st Wahyudi Khoeris Salimi Department of Information Systems Telkom University Bandung, Indonesia wahyudikhoeris@student.telkomuniver sity.ac.id 2nd Tien Fabrianti Kusumasari Department of Information Systems Telkom University Bandung, Indonesia tienkusumasari@telkomuniversity.ac.id l 3rd Sinung Suakanto Department of Information Systems Telkom University Bandung, Indonesia sinungsuakanto@telkomuniversity.ac.i d Abstract— There are serious talent challenges in Indonesia's information sector (IT) arising from the gap between what industry needs and available graduates, compounded by unstructured hiring practices that a


In [ ]:
# A compact, free local embedding model.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

# normalize_embeddings=True makes dot product equivalent to cosine similarity.
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (26, 384)


In [ ]:
TOP_K = 5  # Number of most relevant chunks supplied to the LLM

def retrieve_relevant_chunks(question, top_k=TOP_K):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    similarities = chunk_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = chunks[index].copy()
        item["score"] = float(similarities[index])
        item["source_id"] = f"S{rank}"
        results.append(item)

    return results

In [ ]:
!pip -q install google-genai

from google import genai
from google.genai import types

GEMINI_API_KEY = getpass.getpass("Enter your Google Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

# You can also try: "gemini-flash-latest"
MODEL = "gemini-2.5-flash"

Enter your Google Gemini API key: ··········


In [ ]:
def answer_question(question, top_k=TOP_K):
    retrieved_chunks = retrieve_relevant_chunks(question, top_k=top_k)

    context_blocks = []

    for item in retrieved_chunks:
        context_blocks.append(
            f"[{item['source_id']}] "
            f"Paper: {item['paper']} | Page: {item['page']} | "
            f"Similarity: {item['score']:.3f}\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    system_instruction = """
You are a research-paper question-answering assistant.

Answer using ONLY the supplied research-paper context.
Do not use outside knowledge.

If the answer is not supported by the context, say:
"I could not find this information in the uploaded papers."

Rules:
1. Give a direct and clear answer.
2. Cite every factual claim using source labels such as [S1].
3. Do not invent methods, datasets, findings, limitations, or citations.
4. If sources disagree, explain the disagreement and cite both sources.
5. End with a short Sources section listing cited source labels,
   paper name, and page number.
"""

    prompt = f"""
Question:
{question}

Retrieved research-paper context:
{context}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
            max_output_tokens=1000
        )
    )

    answer = response.text

    if not answer:
        answer = "The model did not return an answer. Please try again."

    return answer, retrieved_chunks

In [ ]:
MODEL = "gemini-3.6-flash"
question = "What is the objective of the paper?"

answer, retrieved = answer_question(question)

print(answer)

Based on the provided context, the primary objectives of the paper are:

* **Prototype Development and Hiring Practice Improvement:** To create a working prototype and provide insights for improving technology-driven hiring practices using the Design Science Research Methodology (DSRM) framework [S5].
* **Integration of LLM-based AI and Workflow Automation:** To demonstrate how Large Language Model (LLM)-based Artificial Intelligence can be integrated with workflow automation tools (specifically the n8n platform) for recruitment workflows [S4, S5].
* **Automated Extraction and Assessment:** To automatically extract candidate details (such as PDF resumes and job description text) and utilize a weighted scoring system to objectively evaluate a candidate's fit for a job [S5].
* **Evaluating System Performance:** To test and compare the automated system's output against manual recruitment procedures in terms of effectiveness, efficiency, skill-matching accuracy, and time savings [S2, S5].


In [ ]:
for item in retrieved:
    print(
        f"\n[{item['source_id']}] {item['paper']} — "
        f"Page {item['page']} "
        f"(Similarity: {item['score']:.3f})"
    )
    print(item["text"][:600], "...")


[S1] IEEE-Research-Paper.pdf — Page 5 (Similarity: 0.234)
Future development should focus on domain-specific AI models for Indonesia's IT sector and increased architectural flexibility to extend applicability beyond IT in order to guarantee that the system can adjust to complex labor market demands and ultimately transform recruitment in Indonesia through a balanced integration of state-of-the-art technology, robust design, and proactive risk management. REFERENCES [1] “Kebutuhan Pekerja IT Indonesia Hampir Capai 2 Juta pada 2025 – Fakultas Teknologi Informasi Universitas Kristen Maranatha.” Diakses: 13 Juni 2025. [Daring]. Tersedia pada: https://it. ...

[S2] IEEE-Research-Paper.pdf — Page 4 (Similarity: 0.221)
by 87.5% when compa ring the two procedures. By saving time, the HR department can increase productivity and concentrate on more strategic duties. E. The Impact of Implementing the N8N Output System on Recruitment The n8n workflow automation -based hiring system produces accur

In [ ]:
while True:
    question = input("\nAsk a question about the uploaded papers (or type 'exit'): ")

    if question.lower().strip() in {"exit", "quit"}:
        print("Session ended.")
        break

    answer, _ = answer_question(question)

    print("\nAnswer:\n")
    print(answer)


Ask a question about the uploaded papers (or type 'exit'): What methodology was used?

Answer:

This study utilized the **Design Science Research Methodology (DSRM)**, a framework in information systems research that comprises six sequential phases [S1], [S3]. 

The specific methodology steps and implementations described in the context include:

* **Design and Development Phase:** A modular artifact was created by combining the low-code platform N8N (as a workflow orchestrator) with Large Language Model (LLM)-based AI agents to automatically extract skills from resumes and job postings and apply a customizable weighted scoring system [S1].
* **Presentation Phase:** The functioning of the system from intake to analytics was described using case studies to ensure real-world efficiency and component compatibility [S1], [S3].
* **Evaluation Phase:** System performance was evaluated by comparing its automated output against manual processing, specifically measuring effectiveness, efficien